In [0]:
%run /Workspace/Users/marcoaurelioreislima@gmail.com/databricks-playground/projects/data_generator/streaming/0_data_generator

In [0]:
def generate_orders_file():
    BASE_PATH = "/Volumes/prd/demo_volumes/rand_engine_data/logs"
    FILE_NAME = "orders"
    FILE_EXT =  "parquet"

    file_generator = FilesGenerator(FakeOrders()).setup_output(BASE_PATH, file_name=FILE_NAME, ext=FILE_EXT)
    #file_generator.delete_files()
    file_generator.write_file(size=1000)
    file_generator.list_files()
    df = spark.read.format(FILE_EXT).load(f"{BASE_PATH}/{FILE_NAME}/{FILE_EXT}")
    df.display()
generate_orders_file()

# FilesGenerator(FakeOrders()).generate_sample(100)

### 1. Prototype Batch Mode

Explore the dataset and test out transformation logic using batch dataframes

In [0]:

schema_path = f"{BASE_PATH}/_schemas"


streaming_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(BASE_PATH)
    .filter(col("traffic_source") == "email")
    .withColumn("mobile", col("device").isin("IOS", "Android"))
    .withColumnRenamed("created_at", "event_timestamp")
    .select("user_id", "event_timestamp", "device", "mobile")
    .writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", schema_path)
    .toTable("prd.l_bronze.email_traffic")
    .awaitTermination()
)
# Wait a moment to let the stream initialize
# You can monitor the stream with query.status or query.lastProgress

In [0]:
%sql
SELECT * FROM prd.l_bronze.email_traffic

In [0]:
from pyspark.sql.functions import window, sum, col
from pyspark.sql.types import TimestampType


parsed_df = (
    spark.readStream
    .table("prd.l_bronze.email_traffic")
    .withWatermark(eventTime="event_timestamp", delayThreshold="10 minutes")
    .groupBy(window(timeColumn="event_timestamp", windowDuration="20 minutes"))
    .agg(sum(col("mobile")).alias("mobile"))
)

In [0]:
%sql
-- Row level security
CREATE FUNCTION hide_rows (region STRING)
RETURN IF(IS_MEMBER('admin'), true, region="US");

ALTER TABLE sales SET ROW FILTER hide_rows ON region;

CREATE OR REPLACE FUNCTION ssn_mask(ssn STRING)
RETURN CASE WHEN IS_MEMBER('admin') THEN ssn ELSE '*****' END;
ALTER TABLE users ALTER COLUMN table_ssn SET MASK ssn_mask;


In [0]:
%sql
-- What tables are in the catalog
SELECT table_name
FROM system.information_schema.tables
WHERE table_catalog = 'prd';

-- Who laste updated the gold tables and when
SELECT table_name, last_altered_by, last_altered
FROM system.information_schema.tables
WHERE table_schema = 'l_bronze'
ORDER BY 1, 3 DESC;

-- Who has access to this table
SELECT table_name
FROM system.information_schema.table_privileges
WHERE table_name = 'sm_customers';

-- Who owns this gold table
SELECT table_owner
FROM system.information_schema.tables
WHERE table_catalog = 'retail_prod'
AND table_schema = 'l_silver'
AND table_name = 'sm_customers';


-- Reference http://docs.databricks.com/en/system-tables/index.html

In [0]:
%sql
-- What is the daily trend in DBU comsumption
SELECT usage_date AS `Date`, SUM(usage_quantity) AS `DBU Consumed`
FROM system.billing.usage
GROUP BY usage_date
ORDER BY usage_date ASC;

-- How many DBUs of each SKU have been used so far this month
SELECT sku_name AS `SKU`, SUM(usage_quantity) AS `DBUs`
FROM system.billing.usage
WHERE month(usage_date) = month(CURRENT_DATE)
GROUP BY sku
ORDER BY `DBUs` DESC;

-- Which 10 users consumed the most DBUs
SELECT identity_metadata.run_as AS `User`, SUM(usage_quantity) AS `DBUs`
FROM system.billing.usage
GROUP BY identity_metadata.run_as 
ORDER BY `DBUs` DESC
LIMIT 10;

-- Which Jobs consumed the most DBUs
SELECT usage_metadata.job_id AS `Job ID`, SUM(usage_quantity) AS `DBUs`
FROM system.billing.usage
GROUP BY identity_metadata.run_as 
ORDER BY `Job ID`;

-- Reference http://docs.databricks.com/en/system-tables/billing.html